In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DecimalType, IntegerType, DateType

In [0]:
%run /Workspace/Users/ashutoshp2@icloud.com/ecommerce_data_pipeline/resources/utils

In [0]:
configs = parse_config_from_json("/Workspace/Users/ashutoshp2@icloud.com/ecommerce_data_pipeline/config/ingestion_configs.json")

In [0]:
products_df = spark.read.table("az_ci_de_dbs.ecom_dev.stg_products")
customers_df = spark.read.table("az_ci_de_dbs.ecom_dev.stg_customers")
orders_df = spark.read.table("az_ci_de_dbs.ecom_dev.stg_orders")

In [0]:
# ---------------------------------------------------------------------------
# Trasformation: Dim Tables
# Data Quality enforcement:
#   - Schema enforcement
#   - Genrate surrogate keys for unique products
#   - Remove product id having nulls
#   - Remove duplicates at dataset level
#   - Missing categorical fields default to 'unknown'; missing price to 0
# ---------------------------------------------------------------------------


def transform_dim_table(
    df: DataFrame,
    module_name: str,
    key_column: str
):
    struct_schema = configs[module_name]['struct_schema']
    schema = parse_struct_schema(struct_schema)
    
    defaults = configs[module_name]['defaults']
    module = configs[module_name]['module']

    
    try:
        logger.info(f"[{module_name}] Removing nulls for {key_column}")
        cleaned_df = remove_nulls_for_key(df, key_column)
    except Exception as e:
        logger.error(f"[{module_name}] Error in remove_nulls_for_key: {e}")
        raise
    try:
        logger.info(f"[{module_name}] Dropping duplicates")
        cleaned_df = cleaned_df.dropDuplicates()
    except Exception as e:
        logger.error(f"[{module_name}] Error in dropDuplicates: {e}")
        raise

    try:
        logger.info(f"[{module_name}] Filling missing values with defaults")
        cleaned_df = fill_defaults(df, defaults)
    except Exception as e:
        logger.error(f"[{module_name}] Error in fill_defaults: {e}")
        raise

    try:
        logger.info(f"[{module_name}] Applying schema enforcement")
        cleaned_df = apply_schema(cleaned_df, schema)
    except Exception as e:
        logger.error(f"[{module_name}] Error in apply_schema: {e}")
        raise
    
    try:
        logger.info(f"[{module_name}] Generating surrogate keys")
        cleaned_df = cleaned_df.withColumn(f"{module}_key", F.monotonically_increasing_id())
    except Exception as e:
        logger.error(f"[{module_name}] Error in withColumn {module}_key: {e}")
        raise
    
    return cleaned_df

In [0]:
# ---------------------------------------------------------------------------
# Transformation: Fact Orders
# Data Quality enforcement:
#   - Schema enforcement
#   - Generate surrogate keys for unique orders
#   - Remove order_id having nulls
#   - parse date formats for order date and ship date
#   - Remove duplicates at dataset level
#   - Missing categorical fields default to 'unknown'; missing price to 0
# ---------------------------------------------------------------------------

def transform_fact_orders(
    df: DataFrame,
    module_name: str,
    key_column: list,
    date_cols: dict,
):
    struct_schema = configs[module_name]['struct_schema']
    schema = parse_struct_schema(struct_schema)
    
    defaults = configs[module_name]['defaults']
    module = configs[module_name]['module']

    try:
        logger.info(f"[{module_name}] Removing nulls for {key_column}")
        cleaned_df = remove_nulls_for_key(df, key_column)
    except Exception as e:
        logger.error(f"[{module_name}] Error in remove_nulls_for_key: {e}")
        raise
    try:
        logger.info(f"[{module_name}] Dropping duplicates")
        cleaned_df = cleaned_df.dropDuplicates()
    except Exception as e:
        logger.error(f"[{module_name}] Error in dropDuplicates: {e}")
        raise

    try:
        logger.info(f"[{module_name}] Filling missing values with defaults")
        cleaned_df = fill_defaults(cleaned_df, defaults)
    except Exception as e:
        logger.error(f"[{module_name}] Error in fill_defaults: {e}")
        raise

    try:
        logger.info(f"[{module_name}] Parsing dates")
        cleaned_df = parse_dates(cleaned_df, date_cols)
    except Exception as e:
        logger.error(f"[{module_name}] Error in parse_dates: {e}")
        raise

    try:
        logger.info(f"[{module_name}] Applying schema enforcement")
        cleaned_df = apply_schema(cleaned_df, schema)
    except Exception as e:
        logger.error(f"[{module_name}] Error in apply_schema: {e}")
        raise
    
    try:
        logger.info(f"[{module_name}] Generating surrogate keys")
        cleaned_df = cleaned_df.withColumn(f"{module}_key", F.monotonically_increasing_id())
    except Exception as e:
        logger.error(f"[{module_name}] Error in withColumn {module}_key: {e}")
        raise
    
    return cleaned_df

In [0]:
def main_transformation():
    try:
        dim_products = transform_dim_table(
            df=products_df,
            module_name='Products',
            key_column='product_id'
        )
        delta_writer(dim_products, 'az_ci_de_dbs.ecom_dev.dim_products')
        dim_customers = transform_dim_table(
            df=customers_df,
            module_name='Customers',
            key_column='customer_id'
        )
        delta_writer(dim_customers, 'az_ci_de_dbs.ecom_dev.dim_customers')
        logger.info("Dim tables written successfully.")
        fact_orders = transform_fact_orders(
            df=orders_df,
            module_name='Orders',
            key_column=['order_id', 'customer_id', 'product_id'],
            date_cols={'order_date': 'd/M/yyyy', 'ship_date': 'd/M/yyyy'}
        )
        delta_writer(fact_orders, 'az_ci_de_dbs.ecom_dev.fact_orders')
        logger.info("Fact table written successfully.")
    except Exception as e:
        logger.error(f"Error in main: {e}")
        raise
